### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from rnanorm import TMM
from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)

## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and raw gene-level counts.

2. **Robust-expression filter (defined on raw counts)** — Keep genes with the 75th percentile of raw counts across samples \(\ge 100\) (i.e., expressed at \(\ge 100\) counts in at least 25% of samples).

3. **TMM normalization** — Apply TMM to the full gene count matrix to compute normalization factors correcting for library size and RNA composition; obtain normalized counts.

4. **Apply gene filter and log-transform** — Subset to the robustly expressed genes and apply \(\log(x+1)\).

5. <strike>**Z-score standardization** — For each gene, center and scale across samples to mean 0 and standard deviation 1.</strike>

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)


clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

raw_counts = df[gene_cols]
print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))

# create mask of robustly expressed genes
robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 100

# normalize counts
df[gene_cols] = TMM().set_output(transform="pandas").fit_transform(df[gene_cols])

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]

# # log transform and standardize (genes only)
df[gene_cols] = np.log(df[gene_cols] + 1)
# df[gene_cols] = (df[gene_cols] - df[gene_cols].mean(axis=0)) / df[gene_cols].std(axis=0)

print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("_raw_counts_NCBI", "_preprocessed")


Before preprocessing:
 - n_genes: 39376



After preprocessing:
 - n_genes: 16907


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_25392/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(



After protein-coding filter:
 - n_genes: 14970


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))

# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)

In [5]:
df_[ordered_cols]

,RFS (months),RFS Censor,Patient ID,RNA-Seq ID,Date of Esophageal Surgery,Relapse,Date of Relapse or Censoring,Date of Death,B cells naive,B cells memory,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,26.1184,0,B012-0001,SL462108,2016-04-01,No,2018-06-04,NaN,0.022107,0.000000,...,3.662069,2.487979,3.810792,3.733372,1.791442,2.494363,4.520041,4.071395,2.996526,3.639040
1,24.5724,0,B012-0004,SL462135,2016-06-10,No,2018-06-27,NaN,0.019443,0.000000,...,2.473485,1.498157,2.663531,2.518551,1.111765,1.356405,3.285345,3.232526,2.252723,2.644813
2,14.0461,1,B012-0007,SL446519,2016-08-01,Yes,2017-10-02,NaN,0.037155,0.000000,...,4.116829,3.025456,4.343368,4.213837,2.025170,2.679345,4.979327,4.475582,3.544341,4.142523
3,17.1382,0,B012-0008,SL462109,2016-10-14,No,2018-03-19,NaN,0.010599,0.000000,...,2.842831,1.888814,3.229611,3.045209,1.598104,2.053633,3.996419,3.611956,2.599311,3.154038
4,24.9671,1,B012-0011,SL446521,2016-12-19,No,2019-01-17,2019-01-17,0.030652,0.002470,...,2.953496,2.304290,3.511578,3.084574,1.602120,2.004306,4.262906,4.016896,3.214875,3.300263
5,40.3618,1,B012-0012,SL462146,2016-12-23,No,2020-05-03,2020-05-03,0.039712,0.000000,...,3.178106,2.342066,3.510186,3.519511,1.735369,2.185630,4.262033,3.852349,2.754341,3.679891
6,3.1579,1,B012-0013,SL446522,2017-01-04,Yes,2017-04-10,2018-03-13,0.014226,0.000000,...,2.769685,2.051023,3.215385,2.916877,1.297129,1.770608,4.029308,3.777015,2.954938,3.271966
7,19.2105,0,B012-0015,SL446523,2017-02-10,No,2018-09-17,NaN,0.012786,0.049479,...,4.296784,3.383753,4.808848,4.524837,2.720960,2.957154,5.564122,5.439549,4.467917,4.613091
8,25.2961,1,B012-0017,SL446534,2017-02-16,Yes,2019-03-27,NaN,0.017473,0.000000,...,4.347126,3.338619,4.469165,4.262920,2.415470,3.063386,5.299694,4.972312,4.179588,4.450961
9,8.0921,1,B012-0018,SL462114,2017-03-08,Yes,2017-11-09,2018-10-20,0.012929,0.001770,...,1.399134,0.482057,1.178963,1.406564,0.491343,0.613028,1.981616,1.776629,1.106709,1.453556
